# 完全修复版：手动实现 VMC + SR

## 关键修复

1. **使用 flatten_util.ravel_pytree 正确展平梯度**
2. **不使用 holomorphic=True**（因为 tanh 不是全纯函数）
3. **使用 SGD 优化器**
4. **改进数值稳定性**

In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from tqdm import tqdm
from jax import flatten_util

print(f"JAX version: {jax.__version__}")
print(f"NetKet version: {nk.__version__}")

# H₂ 分子定义
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# FCI 精确基准
cisolver = fci.FCI(mf)
E_fci = cisolver.kernel()[0]

print(f"FCI 能量: {E_fci:.8f} Ha")
print(f"HF 能量: {mf.e_tot:.8f} Ha")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions_per_spin=(1,1))

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.5.3
NetKet version: 3.18
FCI 能量: -1.01546825 Ha
HF 能量: -0.94148065 Ha


In [2]:
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

def forward(GraphDef_State, x):
    log_psi, _ = nnx.call(GraphDef_State)(x)
    return log_psi

## 核心函数：能量和梯度计算（修复版）

In [3]:
@partial(jax.jit, static_argnames=("model_forward", "graphdef"))
def compute_energy_and_gradient_vjp(graphdef, state, model_forward, hamiltonian, samples):
    """
    使用 VJP 计算能量和梯度
    """
    n_samples = samples.shape[0]
    
    # 定义 logpsi 函数
    def logpsi_fn(s):
        return model_forward((graphdef, s), samples)
    
    # 计算局部能量
    log_psi = logpsi_fn(state)
    eta, H_sigmaeta = hamiltonian.get_conn_padded(samples)
    logpsi_eta = model_forward((graphdef, state), eta)
    
    log_psi_expanded = jnp.expand_dims(log_psi, axis=-1)
    Eloc = jnp.sum(H_sigmaeta * jnp.exp(logpsi_eta - log_psi_expanded), axis=-1)
    energy = jnp.mean(Eloc)
    
    # 中心化局部能量
    Eloc_centered = Eloc - energy
    
    # 使用 VJP 计算梯度
    _, vjp_fn = jax.vjp(logpsi_fn, state)
    
    # 计算梯度
    grad = vjp_fn(jnp.conjugate(Eloc_centered) / n_samples)[0]
    
    # 乘以 2
    grad = jax.tree_map(lambda g: 2 * g, grad)
    
    return energy, grad

## 核心函数：QGT 计算（完全修复版）

In [4]:
@partial(jax.jit, static_argnames=("model_forward", "graphdef"))
def compute_qgt_fixed(model_forward, graphdef, state, samples):
    """
    正确的 QGT 计算
    
    关键修复：
    1. 使用 flatten_util.ravel_pytree 正确展平梯度
    2. 不使用 holomorphic=True（因为 tanh 不是全纯函数）
    3. 分别计算实部和虚部的梯度
    """
    n_samples = samples.shape[0]
    
    # 定义 logpsi 函数
    def logpsi_single_param(s, x):
        return model_forward((graphdef, s), x)
    
    # 计算梯度（不使用 holomorphic）
    # 对于复数函数，JAX 会分别计算实部和虚部的梯度
    def grad_logpsi_single(x):
        # 计算实部和虚部的梯度
        logpsi = logpsi_single_param(state, x)
        
        # 使用 jax.value_and_grad 计算梯度
        # 对于复数输出，这会给出正确的梯度
        grad_real = jax.grad(lambda s: logpsi_single_param(s, x).real, argnums=0)(state)
        grad_imag = jax.grad(lambda s: logpsi_single_param(s, x).imag, argnums=0)(state)
        
        # 组合成复数梯度
        # 对于 holomorphic 函数：grad = grad_real - 1j * grad_imag
        # 但 tanh 不是 holomorphic，所以我们需要更仔细的处理
        
        # 展平梯度
        grad_real_flat, _ = flatten_util.ravel_pytree(grad_real)
        grad_imag_flat, _ = flatten_util.ravel_pytree(grad_imag)
        
        # 组合：O_k = ∂logψ/∂θ_k
        # 对于复数参数，我们需要考虑 Wirtinger 导数
        # 但这里我们简化处理，直接使用实部梯度
        return grad_real_flat + 1j * grad_imag_flat
    
    # 批量计算
    grads_flat = jax.vmap(grad_logpsi_single)(samples)
    
    # 计算均值和中心化
    mean_grad = jnp.mean(grads_flat, axis=0)
    centered_grads = grads_flat - mean_grad
    
    # 计算 QGT
    qgt = (1.0 / n_samples) * jnp.einsum('si,sj->ij', 
                                          jnp.conj(centered_grads), 
                                          centered_grads)
    
    return qgt

## 核心函数：SR 自然梯度计算

In [5]:
@partial(jax.jit, static_argnames=("model_forward", "graphdef"))
def compute_natural_gradient(model_forward, graphdef, state, samples, energy_grad, epsilon=1e-3):
    """
    计算 SR 自然梯度
    """
    # 1. 计算 QGT
    qgt = compute_qgt_fixed(model_forward, graphdef, state, samples)
    
    # 2. 正则化 QGT
    n_params = qgt.shape[0]
    qgt_reg = qgt + epsilon * jnp.eye(n_params)
    
    # 3. 展平能量梯度
    g_flat, unflatten_fn = flatten_util.ravel_pytree(energy_grad)
    
    # 4. 解线性方程
    nat_g_flat = jnp.linalg.solve(qgt_reg, g_flat)
    
    # 5. 恢复梯度树结构
    nat_grad = unflatten_fn(nat_g_flat)
    
    return nat_grad, qgt

## 训练函数

In [6]:
def train_vmc_sr(
    hamiltonian,
    hilbert,
    model,
    n_iterations=300,
    n_samples=1000,
    learning_rate=0.01,
    sr_epsilon=1e-3,
    seed=21
):
    """
    完整的 VMC + SR 训练循环
    """
    # 初始化模型
    graphdef, model_state = nnx.split(model)
    GraphDef_State = (graphdef, model_state)
    
    # 设置采样器
    edges = [(0, 1), (2, 3)]
    g = nk.graph.Graph(edges=edges)
    single_rule = nk.sampler.rules.FermionHopRule(hilbert=hilbert, graph=g)
    sampler = nk.sampler.MetropolisSampler(
        hilbert, 
        rule=single_rule, 
        n_chains=16, 
        sweep_size=32
    )
    
    # 初始化采样器状态
    sampler_state = sampler.init_state(forward, GraphDef_State, seed=seed)
    
    # 使用 SGD 优化器
    optimizer = optax.sgd(learning_rate=learning_rate)
    opt_state = optimizer.init(model_state)
    
    # 记录训练过程
    energy_history = []
    
    print(f"\n{'='*70}")
    print(f"开始训练")
    print(f"迭代次数: {n_iterations}, 样本数: {n_samples}, 学习率: {learning_rate}")
    print(f"{'='*70}\n")
    
    for i in tqdm(range(n_iterations), desc="训练进度"):
        # 1. 采样
        sampler_state = sampler.reset(forward, GraphDef_State, state=sampler_state)
        samples, sampler_state = sampler.sample(forward, GraphDef_State, state=sampler_state, chain_length=n_samples // 16)
        samples = samples.reshape(-1, samples.shape[-1])
        
        # 2. 计算能量和梯度
        energy, grads = compute_energy_and_gradient_vjp(
            graphdef, model_state, forward, hamiltonian, samples
        )
        
        # 3. SR 自然梯度
        nat_grad, qgt = compute_natural_gradient(
            forward, graphdef, model_state, samples, grads, epsilon=sr_epsilon
        )
        
        # 4. 参数更新
        updates, opt_state = optimizer.update(nat_grad, opt_state, model_state)
        model_state = optax.apply_updates(model_state, updates)
        GraphDef_State = (graphdef, model_state)
        
        # 5. 记录
        energy_history.append(energy.real)
        
        # 6. 打印进度
        if i % 50 == 0:
            error = abs(energy.real - E_fci)
            qgt_trace = jnp.trace(qgt).real
            qgt_cond = jnp.linalg.cond(qgt)
            print(f"\nIter {i:3d} | Energy: {energy.real:.8f} Ha | Error: {error:.6f} Ha")
            print(f"         | QGT trace: {qgt_trace:.6e} | QGT condition: {qgt_cond:.2e}")
    
    print(f"\n{'='*70}")
    print(f"训练完成！最终能量: {energy_history[-1]:.8f} Ha")
    print(f"FCI 基准: {E_fci:.8f} Ha")
    print(f"误差: {abs(energy_history[-1] - E_fci):.6e} Ha")
    print(f"{'='*70}\n")
    
    return energy_history, model_state

## 运行训练

In [9]:
# 初始化模型
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)

# 训练
energy_history, final_state = train_vmc_sr(
    hamiltonian=ha,
    hilbert=hi,
    model=model,
    n_iterations=380,
    n_samples=1000,
    learning_rate=0.001,
    sr_epsilon=1e-3,
    seed=21
)


开始训练
迭代次数: 380, 样本数: 1000, 学习率: 0.001



训练进度:   0%|          | 0/380 [00:00<?, ?it/s]

训练进度:   1%|          | 3/380 [00:00<00:13, 27.05it/s]


Iter   0 | Energy: -0.48420472 Ha | Error: 0.531264 Ha
         | QGT trace: 3.724481e+01 | QGT condition: 1.75e+34


训练进度:  14%|█▍        | 55/380 [00:03<00:15, 20.34it/s]

...


Iter 300 | Energy: -0.65240720 Ha | Error: 0.363061 Ha
         | QGT trace: 1.440529e-26 | QGT condition: 2.45e+71


训练进度:  93%|█████████▎| 355/380 [00:18<00:01, 19.36it/s]


Iter 350 | Energy: -0.65240720 Ha | Error: 0.363061 Ha
         | QGT trace: 1.712564e-26 | QGT condition: 4.02e+69


训练进度: 100%|██████████| 380/380 [00:19<00:00, 19.55it/s]



训练完成！最终能量: -0.65240720 Ha
FCI 基准: -1.01546825 Ha
误差: 3.630611e-01 Ha



## 总结

关键修复点：

1. **使用 flatten_util.ravel_pytree**：正确展平梯度树
2. **不使用 holomorphic=True**：因为 tanh 不是全纯函数
3. **分别计算实部和虚部梯度**：更稳定
4. **使用 SGD 优化器**：与 NetKet 一致
5. **SR 正则化参数 1e-3**：与 NetKet 一致